In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.


In [2]:
import cudf
import cupy as cp
import torch

In [3]:
mapping_df = cudf.read_feather("/workspace/data/450_probe_to_feature_mapping.feather")

/opt/conda/envs/rapids-env/lib/python3.13/site-packages/cudf/io/feather.py:16: UserWarning: Using CPU via PyArrow to read feather dataset, this may be GPU accelerated in the future
  warnings.warn(


In [4]:
m_values_df = cudf.read_feather("/workspace/data/GSE111629_data_test.feather")

In [5]:
mapping_df = mapping_df.set_index("Probe_ID")

`m_values_df` has probes as cols and samples as rows. The mapping to genomic regions needs it the other way around.
Before transposing, the samples need to be set as the dataframe index.

In [6]:
m_values_df = m_values_df.set_index("Sample_Name")

In [7]:
all_cols = m_values_df.columns.to_list()

In [8]:
probe_cols = [c for c in all_cols if c.startswith('cg') or (c.startswith('ch') and not c.startswith('cha'))]

In [9]:
probe_set = set(probe_cols)
pheno_cols = [c for c in all_cols if c not in probe_set]

In [10]:
pheno_df = m_values_df[pheno_cols]
m_values_wide = m_values_df[probe_cols]

In [11]:
m_values_df = m_values_wide.T
m_values_df.index.name = "Probe_ID"

In [12]:
merged_df = m_values_df.join(mapping_df, how="inner")

In [13]:
grouped = merged_df.groupby("Feature_Name")

In [14]:
mean_df = grouped.mean()
var_df = grouped.var()

In [15]:
var_df = var_df.fillna(0.0)

In [16]:
ordered_regions = mean_df.index.to_pandas().tolist()

In [18]:
import cudf
import torch

patient_graphs = {}
sample_ids = mean_df.columns.tolist()

for sample in sample_ids:
    patient_means = mean_df[sample].values
    patient_vars = var_df[sample].values
    
    # cupy arrays implement the __dlpack__ protocol directly; toDlpack() is deprecated and yields single-use capsules
    tensor_means = torch.utils.dlpack.from_dlpack(patient_means).to(torch.float32)
    tensor_vars = torch.utils.dlpack.from_dlpack(patient_vars).to(torch.float32)
    
    # 4. Stack horizontally [N, 2]
    X = torch.stack((tensor_means, tensor_vars), dim=1)
    
    patient_graphs[sample] = X

print(f"Shape of X tensor: {patient_graphs[sample_ids[0]].shape}")

Shape of X tensor: torch.Size([90399, 2])


In [19]:
n_samples = len(sample_ids)
n_regions, n_feats = X.shape
total_bytes = n_samples * n_regions * n_feats * 4
print(f"samples={n_samples}, regions={n_regions}, feats={n_feats}, total size={total_bytes / 1024**2:.1f} MiB")

samples=536, regions=90399, feats=2, total size=369.7 MiB


## Persist patient graphs to disk

All patient tensors share the same shape and row order (`ordered_regions`), so they are stacked into a single `[n_samples, n_regions, 2]` tensor and saved with `safetensors` (fast mmap-based load, no pickle deserialization risk unlike `torch.save`). `sample_ids` and `ordered_regions` are saved alongside as JSON metadata since safetensors only stores tensors + string key/value pairs.


In [ ]:
import json
from pathlib import Path
from safetensors.torch import save_file

out_dir = Path("/workspace/results")
out_dir.mkdir(parents=True, exist_ok=True)

# stack in sample_ids order -> [n_samples, n_regions, 2], contiguous for safetensors
stacked = torch.stack([patient_graphs[s] for s in sample_ids], dim=0).contiguous().cpu()

save_file({"patient_graphs": stacked}, out_dir / "patient_graphs.safetensors")
with open(out_dir / "patient_graphs_meta.json", "w") as f:
    json.dump({"sample_ids": sample_ids, "ordered_regions": ordered_regions}, f)

print(f"Saved tensor {tuple(stacked.shape)} to {out_dir}")


Saved tensor (536, 90399, 2) to /workspace/data/patient_graphs


## Reload persisted graphs and continue

To resume from disk you only need the `.safetensors` file, its JSON metadata sidecar, and the `safetensors`/`torch` packages (no cudf/cupy pipeline re-run required). `load_file` mmaps the tensor straight from disk; `.to("cuda")` moves it to GPU only when needed downstream.


In [ ]:
import json
from pathlib import Path
from safetensors.torch import load_file

load_dir = Path("/workspace/results")

with open(load_dir / "patient_graphs_meta.json") as f:
    meta = json.load(f)
loaded_sample_ids = meta["sample_ids"]
loaded_ordered_regions = meta["ordered_regions"]

loaded_stacked = load_file(load_dir / "patient_graphs.safetensors")["patient_graphs"]

# rebuild the per-patient dict keyed the same way as the in-memory patient_graphs
loaded_patient_graphs = {s: loaded_stacked[i] for i, s in enumerate(loaded_sample_ids)}

print(f"Loaded {loaded_stacked.shape} for {len(loaded_sample_ids)} samples, {len(loaded_ordered_regions)} regions")
